# 02. Prompt Engineering 실습
**SK하이닉스 Autonomous R&D — Day 3** 

---
## 0. 준비

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))


def ask(system_prompt, user_prompt, temperature=0.2):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
    )
    return response.choices[0].message.content

In [ ]:
# 기본적으로 같은 모델이라도 프롬프트에 따라 다른 출력을 생성

system = 'You are a helpful assistant. Answer in Korean.'

prompts_ko = [
    "프랑스의 수도는",
    "Q: 프랑스의 수도는 어디인가요? A:",
    "프랑스의 수도 도시는",
]

for prompt in prompts_ko:
    print(f"프롬프트: {prompt}")
    print(f"생성: {ask(system, prompt, temperature=0.2)}")
    print()

---
## 1. Zero-shot vs Few-shot 

| 방식 | 설명 |
|------|------|
| **Zero-shot** | 예시 없이 지시만 |
| **Few-shot** | 원하는 형식의 **예시**를 함께 제공 |

In [ ]:
system = 'You are a helpful assistant. Answer in Korean.'
# Zero-shot — 형식 지시 없음
user_zero = '''
아래 영화 리뷰가 긍정인지 부정인지 판정해줘.
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Zero-shot ===')
print(ask(system, user_zero))

In [ ]:
# Few-shot — 출력 형식 예시 포함
user_few = '''
아래 형식으로 감성 판정해줘.

[예시]
입력: "배우 연기가 훌륭했다" → 긍정 | 9/10
입력: "스토리가 지루했다" → 부정 | 3/10

[실제 데이터]
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Few-shot ===')
print(ask(system, user_few))

In [ ]:
zero_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

문제: 행복한 : 슬픈 :: 좋은 :"""

print(ask(system, zero_shot_prompt_ko, temperature=0.7))

In [ ]:
few_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 좋은 :"""

print('Few-shot 프롬프트:')
print(few_shot_prompt_ko)
print('\n생성된 완성:')
print(ask(system, few_shot_prompt_ko, temperature=0.7))

---
## 2. 역할(Role) 설정 

In [ ]:
question = '하루 커피 4잔 마셔도 괜찮을까?'

print('=== 일반 assistant ===')
print(ask('You are a helpful assistant.', question))
print('\n' + '=' * 50 + '\n')

In [ ]:
print('=== 영양사 역할 ===')
print(ask(
    'You are a registered dietitian. Answer in Korean with caffeine mg estimate and health advice.',
    question,
))

In [ ]:
roles = [
    "당신은 친절한 고객 서비스 상담원입니다.",
    "당신은 전문적인 기술 문서 작성자입니다.",
    "당신은 창의적인 소설가입니다."
]

question = "인공지능에 대해 설명해주세요."

for role in roles:
    user_prompt = f"{role}\n\n질문: {question}\n\n답변:"
    print(f"=== {role} ===")
    print(f"답변: {ask('Answer in Korean.', user_prompt, temperature=0.8)}")
    print()

---
## 3. 출력 형식 지정 

In [ ]:
topic = '재택근무의 장단점 3가지'
system_ko = 'Answer in Korean.'

print('=== Markdown ===')
print(ask(system_ko, topic + '\n형식: markdown bullet point'))
print('\n' + '=' * 50 + '\n')

In [ ]:
print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (항목 | 장점 | 단점)만 출력',
))

---
### 출력 형식 (표 / JSON)

In [ ]:
import json

topic = '식각(Etch) 공정에서 수율에 영향 주는 요인 3가지'
system_ko = 'Answer in Korean.'

print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (요인 | 설명 | 대응)만 출력',
))

In [ ]:
print('=== JSON ===')
result = ask(
    'Output ONLY valid JSON. No markdown.',
    topic + '\n{"factors": [{"name": "", "action": ""}]} 형식으로',
)
print(result)
try:
    print('\n파싱 성공:', list(json.loads(result).keys()))
except json.JSONDecodeError:
    print('\n파싱 실패 — ONLY valid JSON 강조 필요')

---
## 4. Chain-of-Thought (CoT)

CoT는 모델이 단계별로 추론 과정을 보여주도록 하는 기법.

In [ ]:
## zero-shot 예시

problem = '''
A팀 10명, B팀 15명, C팀 5명.
전체 25명 중 40% 이상이 A팀이면 A팀에 추가 인원 배치.
지금 추가 배치가 필요한가?
'''
system = 'You are a helpful assistant. Answer in Korean.'
print('=== CoT 없음 ===')
print(ask(system, problem))

In [ ]:
print('=== CoT 적용 ===')
print(ask(
    system + ' Show step-by-step reasoning before final answer.',
    problem + '\n\n단계별로 생각해 봅시다.',
))

In [ ]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

In [ ]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

천천히 단계별로 생각해봅시다.

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

In [ ]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

답: 6000원

---

문제: 한 상점에서 귤 1개와 바나나 12개를 샀습니다. 사과는 개당 1000원, 배는 개당 200원입니다. 총 금액은 얼마인가요?

답: 3400원

---


문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: """

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

In [ ]:
## one-shot 예시

problem = """다음 문제를 단계별로 풀어보세요.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

단계별 풀이:
1. 사과 3개의 가격: 3 × 1000 = 3000원
2. 배 2개의 가격: 2 × 1500 = 3000원
3. 총 금액: 3000 + 3000 = 6000원
답: 6000원

---

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

단계별 풀이:"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

---
## 5. System + User 프롬프트 

In [ ]:
system_prompt = '''
너는 스타트업 PM의 일정 관리 AI다.
규칙: 결론 먼저, bullet 3개, 한국어
'''

user_prompt = '''
이번 주 마감: 월-기획안, 수-코드리뷰, 금-발표.
목요일에 하루 종일 회의가 잡혔어. 우선순위 조정해줘.
'''

print(ask(system_prompt, user_prompt))

In [ ]:
system_md = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤 "
    "리스크와 가정을 명시해야 한다."
)

user_md = "이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘."

answer = ask(system_md, user_md, temperature=0.7)
print(answer)

---
## 6. Self-check Prompting

In [ ]:
check_prompt = f"""아래는 네가 작성한 'VIP 이탈 상위 200명 액션 플랜' 초안이다.

[초안]
{answer}

이제 아래 체크리스트로 점검하고, 필요하면 수정본을 작성하라.

체크리스트:
1) 논리적 모순/비약이 있는가?
2) 실행 불가능하거나 모호한 액션이 있는가? (담당/기한/방법이 불명확한지)
3) 과도한 가정이 있는가?
4) 표 형식이 지켜졌는가? (결론 먼저, 리스크/가정 포함)

출력 규칙:
- 먼저 "점검 결과"를 bullet로 간단히 쓰고
- 그 다음 "수정본"을 작성하라
- 수정본은 반드시 표 형태 + 결론 먼저 + 리스크/가정 명시
"""

final_answer = ask(system_md, check_prompt, temperature=0.7)
print('\n--- Self-check 후 최종 답변(수정본) ---')
print(final_answer)

---
## 7. Constraint Prompting

In [ ]:
system_prompt = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤, "
    "리스크와 가정을 명시해야 한다."
)

constraints = """조건(반드시 준수):
- 추가 예산은 최대 1억 원 이내
- 인력 증원 불가 (기존 인력 내에서 운영)
- 2주 이내 실행 가능한 액션만 제시
- 고객에게 직접 연락하는 액션은 과도한 빈도를 피하고(1회 이내),
  개인정보/컴플라이언스 리스크를 고려할 것
"""

user_prompt = f"""이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘.

{constraints}

출력 형식:
- 결론(한 줄) 먼저
- 그 다음 표로 정리 (컬럼 예: 타깃/액션/채널/우선순위/기간/기대효과/담당)
- 마지막에 리스크/가정 명시
"""

constrained_answer = ask(system_prompt, user_prompt, temperature=0.7)
print('\n--- 조건 기반 최종 답변(답변만) ---')
print(constrained_answer)